In [2]:
import sys, pprint, pandas as pd  
sys.path.append('../../')
sys.path.append('../')
sys.path.append('./')

import re,pandas as pd
import plotly.io as pio
import json
from typing import Any, Dict, List, Iterable, Literal, Union, Optional,TypedDict
from typing_extensions import Self   
from uuid import uuid4
from pydantic import BaseModel, Field 
from get_llm_model import azure_llm_if

from visualization_system.visualization_backend.all_classes import * 



# Initialize 

In [3]:

# these are just mocks 
def get_config():
    return None 

class DataDrivenStorage:
        
    def __init__( self, config_vars ):
        pass 

    def get_project_dataset(self, project_name=None, filters=None):
        #path =  "../datasets/Demo1/"
        path =  Path("../datasets/IX5I_4P/") 

        inj, prod, locs = self.fetch_data(path) 
        return inj, prod, locs

    def fetch_data(self,path:Path):
        inj  = pd.read_csv(path / "injectors.csv")
        pinj = pd.read_csv(path / "producers.csv")
        locs = pd.read_csv(path / "locations.csv")
        inj['DATE'] = pd.to_datetime( inj['DATE'],dayfirst=True)
        inj['DAY']   = inj['DATE'].dt.day
        inj['MONTH'] = inj['DATE'].dt.month
        inj['YEAR']  = inj['DATE'].dt.year
        pinj['DATE'] = pd.to_datetime( pinj['DATE'],dayfirst=True)
        pinj['DAY']   = pinj['DATE'].dt.day
        pinj['MONTH'] = pinj['DATE'].dt.month
        pinj['YEAR']  = pinj['DATE'].dt.year


        return inj, pinj, locs

def initialize_system( llm ):
   

    #IMPORTS
    from visualization_system.visualization_backend.analyst.semantics.semantic_models import SemanticCatalog, semantic_catalog
    from visualization_system.visualization_backend.analyst.semantics.semantic_models import idioms as all_idiom_rules
    vis_system = AgenticSystem( llm )


    idiom = 'duckdb'
    idiom_rules = all_idiom_rules[idiom]
    semantic_catalog_model = SemanticCatalog.model_validate( semantic_catalog )

    analyst = vis_system.data_analyst_component
    analyst.init_semantic_models( semantic_catalog_model,idiom_rules)
    



    return vis_system


llm = azure_llm_if()
vis_system = initialize_system(llm)


# data changes
# this mocks data comming from the UI
# so we just update tge analyst 
inj,prod,locs = DataDrivenStorage( get_config() ).get_project_dataset(123, {}) 
vis_system.data_analyst_component.set_data( {'injectors':inj, 
                                             'producers':prod, 
                                             'locations': locs } )





zero temp, seed 42, top_p = 1


In [4]:

query = """Explain VRR briefly and then 
list the 5 top producers based on the cummulated oil production in 2018,
then show the cummulated liquid production since year 2015 for all the wells 
"""

#this is what the presenter consumes 
#execution_state = vis_system.run( query )



In [5]:
import pickle 
with open("execution_state.pkl", "rb") as file:
    loaded_data = pickle.load(file)

#import pickle
#with open("execution_state.pkl", "wb") as file:
#    pickle.dump(execution_state, file)

execution_state = loaded_data

In [6]:

presenter = PresenterComponent2( llm )
ui_items = presenter.run( execution_state )
ui_items

processing dataframe result
(4, 2)
instruction Identify the top 5 producer wells based on their cumulative oil production in the year 2018. Then, calculate the cumulative liquid production (oil + water) for all wells starting from the year 2015 to the most recent data available. Present the results in a clear format.
Extracting context
In the processor  NAME
-----------here
isnide infer_column_summary, caling role
inferring role for  NAME
is_string_dtype
This is just before the warning error
This is just after the warning error
Role found categorical
Retuning summary
Summary done 
In the processor  cumulative_oil_volume
-----------here
isnide infer_column_summary, caling role
inferring role for  cumulative_oil_volume
is_numeric_dtype
Role found categorical_numeric
Retuning summary
Summary done 
selecting a plan
Entering in select_chart_plan
Leving select_chart_plan 1 
Leving select_chart_plan 2 
{'reason': 'The table contains data for the top 5 producer wells based on cumulative oil pr

PresenterResponse(agent='presenter', layout='vertical', items=[UIItem(id='text_f465f4fc', type='text', title=None, data={'text': 'Voidage Replacement Ratio (VRR) is a key reservoir management metric that measures the ratio of the volume of injected fluids (e.g., water, gas) to the volume of produced reservoir fluids (oil, gas, and water). It is used to assess whether the voidage created by production is being adequately replaced to maintain reservoir pressure and optimize recovery. \n\nA VRR of 1.0 indicates that the injected volume equals the produced volume, helping to maintain pressure. A VRR less than 1.0 may lead to pressure depletion, while a VRR greater than 1.0 could result in over-pressurization.'}, meta={}), UIItem(id='chart_d29a4d11', type='chart', title='Top 5 producer wells 2018', data={'engine': 'plotly', 'plotly': {'data': [{'type': 'table', 'header': {'values': ['Name', 'Cumulative oil volume'], 'align': 'left'}, 'cells': {'values': [['P4', 'P3', 'P1', 'P2'], ['12875.19

In [10]:
print(ui_items.items[0].data['text'])


Voidage Replacement Ratio (VRR) is a key reservoir management metric that measures the ratio of the volume of injected fluids (e.g., water, gas) to the volume of produced reservoir fluids (oil, gas, and water). It is used to assess whether the voidage created by production is being adequately replaced to maintain reservoir pressure and optimize recovery. 

A VRR of 1.0 indicates that the injected volume equals the produced volume, helping to maintain pressure. A VRR less than 1.0 may lead to pressure depletion, while a VRR greater than 1.0 could result in over-pressurization.


In [11]:
ui_items.items[1]

UIItem(id='chart_d29a4d11', type='chart', title='Top 5 producer wells 2018', data={'engine': 'plotly', 'plotly': {'data': [{'header': {'values': ['Name', 'Cumulative oil volume'], 'align': 'left'}, 'cells': {'values': [['P4', 'P3', 'P1', 'P2'], ['12875.1990353', '12768.2343158', '12538.622009199999', '11907.329711700002']], 'align': 'left'}, 'type': 'table'}], 'layout': {'title': {'text': 'Top 5 producer wells by cumulative oil production in 2018'}}, 'config': {'responsive': True, 'displaylogo': False}}}, meta={'description': 'Contains the top 5 producer wells based on cumulative oil production in 2018. Columns include: - name: Producer well identifier - cumulative_oil_volume: Total oil volume produced by the well in 2018'})

In [7]:
item = ui_items.items[1]
item = item.data['plotly']
pio.show(item)
